In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, LSTM, Flatten
import shap
import lime
from lime.lime_tabular import LimeTabularExplainer

# Load datasets
fraud_data = pd.read_csv("Fraud_Data.csv")
ip_data = pd.read_csv("IpAddress_to_Country.csv")
credit_card_data = pd.read_csv("creditcard.csv")



In [ ]:
# Handle missing values
fraud_data.dropna(inplace=True)
credit_card_data.dropna(inplace=True)

# Remove duplicates
fraud_data.drop_duplicates(inplace=True)
credit_card_data.drop_duplicates(inplace=True)



In [ ]:
# Convert IP to integer for merging
def ip_to_int(ip):
    octets = list(map(int, ip.split('.')))
    return (16777216 * octets[0]) + (65536 * octets[1]) + (256 * octets[2]) + octets[3]

fraud_data['ip_int'] = fraud_data['ip_address'].apply(ip_to_int)
ip_data['lower_bound_ip_int'] = ip_data['lower_bound_ip_address'].apply(ip_to_int)
ip_data['upper_bound_ip_int'] = ip_data['upper_bound_ip_address'].apply(ip_to_int)

# Merge Fraud_Data with country info
fraud_data = fraud_data.merge(ip_data, how='left', left_on='ip_int', right_on='lower_bound_ip_int')

# Feature Engineering
def extract_time_features(df, time_col):
    df[time_col] = pd.to_datetime(df[time_col])
    df['hour_of_day'] = df[time_col].dt.hour
    df['day_of_week'] = df[time_col].dt.weekday
    return df

fraud_data = extract_time_features(fraud_data, 'purchase_time')



In [ ]:
# Encode categorical features
le = LabelEncoder()
fraud_data['source'] = le.fit_transform(fraud_data['source'])
fraud_data['browser'] = le.fit_transform(fraud_data['browser'])

# Scale numeric features
scaler = StandardScaler()
numeric_cols = ['purchase_value', 'hour_of_day', 'day_of_week']
fraud_data[numeric_cols] = scaler.fit_transform(fraud_data[numeric_cols])



In [ ]:
# Train-test split
X_fraud = fraud_data.drop(columns=['class'])
y_fraud = fraud_data['class']
X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = train_test_split(X_fraud, y_fraud, test_size=0.2, random_state=42)

X_credit = credit_card_data.drop(columns=['Class'])
y_credit = credit_card_data['Class']
X_train_credit, X_test_credit, y_train_credit, y_test_credit = train_test_split(X_credit, y_credit, test_size=0.2, random_state=42)



In [ ]:
# Model Selection and Training
def train_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(classification_report(y_test, preds))
    return model

models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier()
}

for name, model in models.items():
    print(f"Training {name}...")
    train_model(model, X_train_fraud, y_train_fraud, X_test_fraud, y_test_fraud)



In [ ]:
# Deep Learning Model
cnn_model = Sequential([
    Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(X_train_fraud.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_train_fraud.values.reshape(-1, X_train_fraud.shape[1], 1), y_train_fraud, epochs=10, batch_size=32)

# SHAP Explainability
explainer = shap.TreeExplainer(models['Random Forest'])
shap_values = explainer.shap_values(X_test_fraud)
shap.summary_plot(shap_values, X_test_fraud)
